# ⚖️ CiteJustice - NyayaAnumana & InCaseLawBERT Training on Google Colab
### Official Dataset: `L-NLProc/NyayaAnumana-Classification-Data`
### Paper: *NyayaAnumana & INLegalLlama: Largest Indian Legal Judgment Prediction Dataset (COLING 2025)*

**In this notebook:**
1. Connects to free 16 GB Tesla T4 GPU
2. Authenticates with Hugging Face using your access token
3. Downloads the official **NyayaAnumana** binary court judgment dataset (`binary_dev.zip` / `CJPE_ext_SCI_HCs_dev.csv`)
4. Verifies standard Indian legal columns: `text` (facts/arguments) and `label` (`0 = Dismissed/Rejected`, `1 = Allowed/Accepted`)
5. Fine-tunes `law-ai/InCaseLawBERT` with Mixed Precision (FP16)
6. Evaluates Accuracy & Weighted F1-Score
7. Packages & downloads `InCaseLawBERT_FineTuned.zip` to drop directly into CiteJustice

## Step 1: Verify Hardware Accelerator (Tesla T4 GPU)

In [ ]:
!nvidia-smi

## Step 2: Install Required Libraries

In [ ]:
!pip install -q transformers datasets huggingface_hub accelerate evaluate torch scikit-learn

## Step 3: Authenticate & Download NyayaAnumana Dataset from Hugging Face

In [ ]:
import os
import zipfile
import glob
import pandas as pd
from huggingface_hub import hf_hub_download
from datasets import Dataset

# Your authorized Hugging Face token
HF_TOKEN = "YOUR_HF_TOKEN_HERE"

print("[*] Downloading official NyayaAnumana Binary Dataset via Hugging Face Cloud...")
zip_local_path = hf_hub_download(
    repo_id="L-NLProc/NyayaAnumana-Classification-Data",
    filename="dev/binary/binary_dev.zip",
    repo_type="dataset",
    token=HF_TOKEN
)
print(f"[+] Downloaded zip to: {zip_local_path}")

# Extract zip contents
DATA_DIR = "nyayaanumana_extracted"
os.makedirs(DATA_DIR, exist_ok=True)
with zipfile.ZipFile(zip_local_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

# List extracted CSV files
csv_files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)
print("[+] Extracted CSV files from NyayaAnumana:")
for f in csv_files:
    print("   -> ", os.path.basename(f))

# Load Supreme Court + High Courts dataset (High quality, standard benchmark)
target_csv = next((f for f in csv_files if "SCI_HCs_dev.csv" in f), csv_files[0])
print(f"\n[*] Loading target dataset: {os.path.basename(target_csv)}")
df = pd.read_csv(target_csv)

print(f"[+] Total cases in file: {len(df)}")
print(f"[+] Columns detected: {df.columns.tolist()}")
print("[*] Preview of first case text and label:")
print(f"    Label: {df.iloc[0]['label']}")
print(f"    Text preview: {str(df.iloc[0]['text'])[:200]}...")

## Step 4: Standardize & Split Dataset (Train: 85% | Val: 15%)
Ensures clean binary labels (`0 = Dismissed/Rejected`, `1 = Allowed/Accepted`) and trims text to 1500 chars.

In [ ]:
# Filter out missing or malformed rows
df_clean = df.dropna(subset=["text", "label"]).copy()
df_clean["label"] = df_clean["label"].astype(int)
df_clean["text"] = df_clean["text"].astype(str).str.slice(0, 1500)

# Balance check
label_counts = df_clean["label"].value_counts().to_dict()
print(f"[+] Label Distribution: {label_counts} (0 = Dismissed, 1 = Allowed)")

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df_clean[["text", "label"]])
split_ds = hf_dataset.train_test_split(test_size=0.15, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

print(f"[+] Split complete: {len(train_ds)} Train cases | {len(val_ds)} Validation cases")

## Step 5: Tokenize with InCaseLawBERT

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "law-ai/InCaseLawBERT"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_func(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

print("[*] Tokenizing dataset with InCaseLawBERT tokenizer (max_length=512)...")
train_tokenized = train_ds.map(tokenize_func, batched=True, batch_size=1000)
val_tokenized = val_ds.map(tokenize_func, batched=True, batch_size=1000)
print("[+] Tokenization finished!")

## Step 6: High-Performance GPU Fine-Tuning (Mixed Precision FP16)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./nyaya_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                          # GPU Mixed Precision acceleration
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

print("[*] Commencing Fine-Tuning on Tesla T4...")
trainer.train()

## Step 7: Export & Automatic Browser Download for CiteJustice

In [ ]:
import shutil
from google.colab import files

EXPORT_DIR = "InCaseLawBERT_FineTuned"
model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

shutil.make_archive(EXPORT_DIR, 'zip', EXPORT_DIR)
print(f"[+] Model saved and zipped into {EXPORT_DIR}.zip")
print("[*] Starting automatic download to your PC...")
files.download(f"{EXPORT_DIR}.zip")